# D. 코호트·LTV·리텐션 분석

첫 구매 월(코호트)별로 N개월 후 재구매율·ARPU를 계산하고, 코호트 히트맵으로 시각화합니다.

**사용 데이터**: `data/` (00 실행 후).

## 1. 데이터 로드 및 코호트 정의

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path.cwd() / "data"
for _ in [Path.cwd() / "data", Path.cwd().parent / "예측 대시보드 용 프로젝트" / "data"]:
    if (_ / "orders_delivered.csv").exists():
        DATA_DIR = _
        break
if not (DATA_DIR / "orders_delivered.csv").exists():
    import kagglehub
    _path = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
    orders = pd.read_csv(_path / "olist_orders_dataset.csv")
    orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
    orders = orders[orders["order_status"] == "delivered"].copy()
    customers = pd.read_csv(_path / "olist_customers_dataset.csv")
    order_payments = pd.read_csv(_path / "olist_order_payments_dataset.csv")
    order_payments["payment_type"] = order_payments["payment_type"].replace("not_defined", "unknown")
    print("(data/ 없음 → kagglehub에서 로드)")
else:
    orders = pd.read_csv(DATA_DIR / "orders_delivered.csv")
    customers = pd.read_csv(DATA_DIR / "customers.csv")
    order_payments = pd.read_csv(DATA_DIR / "order_payments.csv")

orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
orders = orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")

# 주문별 결제액
order_value = order_payments.groupby("order_id")["payment_value"].sum().reset_index().rename(columns={"payment_value": "order_value"})
orders = orders.merge(order_value, on="order_id")

# 고객별 첫 주문 월 = 코호트
first_order = orders.groupby("customer_unique_id").agg(
    first_month=("order_purchase_timestamp", "min"),
    order_count=("order_id", "nunique"),
    total_value=("order_value", "sum"),
).reset_index()
first_order["cohort_month"] = first_order["first_month"].dt.to_period("M").dt.to_timestamp()
first_order["cohort_week_start"] = first_order["first_month"].dt.to_period("W-MON").dt.start_time
orders["order_month"] = orders["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
orders["order_week_start"] = orders["order_purchase_timestamp"].dt.to_period("W-MON").dt.start_time
orders = orders.merge(first_order[["customer_unique_id", "cohort_month", "cohort_week_start"]], on="customer_unique_id")
print("코호트 월 수:", first_order["cohort_month"].nunique(), "| 코호트 주 수:", first_order["cohort_week_start"].nunique())

코호트 월 수: 22


## 2. 경과월별 재구매율 (코호트 × 경과월)

In [2]:
first_order["cohort_month"] = pd.to_datetime(first_order["cohort_month"])
orders["cohort_month"] = pd.to_datetime(orders["cohort_month"])
orders["order_month"] = pd.to_datetime(orders["order_month"])
orders["months_since_first"] = (orders["order_month"].dt.year - orders["cohort_month"].dt.year) * 12 + (orders["order_month"].dt.month - orders["cohort_month"].dt.month)
orders["weeks_since_first"] = (orders["order_week_start"] - orders["cohort_week_start"]).dt.days // 7

# 코호트별·경과월별 재구매 (해당 월에 1건 이상 주문한 고객 비율)
cohort_sizes = first_order.groupby("cohort_month").size()
retention_data = []
for (cohort, month_num), grp in orders.groupby(["cohort_month", "months_since_first"]):
    n_customers = grp["customer_unique_id"].nunique()
    cohort_size = cohort_sizes.get(cohort, 0)
    if cohort_size > 0:
        retention_data.append({"cohort": cohort, "month_num": month_num, "retention_pct": n_customers / cohort_size * 100, "n": n_customers})

ret_df = pd.DataFrame(retention_data)
if len(ret_df) > 0:
    pivot_ret = ret_df.pivot(index="cohort", columns="month_num", values="retention_pct")
    print("리텐션 행렬 (코호트 × 경과월, %):")
    print(pivot_ret.iloc[:5, :8].round(1))

ValueError: Unit M is not supported. Only unambiguous timedelta values durations are supported. Allowed units are 'W', 'D', 'h', 'm', 's', 'ms', 'us', 'ns'

## 2.5 주간 코호트 리텐션 (대시보드용)

**cohort_week_start**, **weeks_since_first** 기준으로 주별 리텐션을 집계합니다.

In [ ]:
cohort_sizes_week = first_order.groupby("cohort_week_start").size()
ret_week_data = []
for (cohort_w, week_num), grp in orders.groupby(["cohort_week_start", "weeks_since_first"]):
    n_customers = grp["customer_unique_id"].nunique()
    cohort_size = cohort_sizes_week.get(cohort_w, 0)
    if cohort_size > 0 and week_num >= 0:
        ret_week_data.append({"cohort_week_start": cohort_w, "weeks_since_first": week_num, "retention_pct": n_customers / cohort_size * 100, "n": n_customers})
ret_week_df = pd.DataFrame(ret_week_data)
if len(ret_week_df) > 0:
    pivot_ret_week = ret_week_df.pivot(index="cohort_week_start", columns="weeks_since_first", values="retention_pct")
    print("주간 리텐션 행렬 (일부):")
    print(pivot_ret_week.iloc[:5, :8].round(1))
else:
    print("주간 리텐션 데이터 없음")

## 3. 코호트 히트맵 시각화

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(ret_df) > 0 and pivot_ret.size > 0:
    plt.figure(figsize=(12, 6))
    sns.heatmap(pivot_ret.fillna(0), annot=False, fmt=".0f", cmap="YlGnBu")
    plt.title("코호트별 리텐션 (경과월)")
    plt.xlabel("경과 월")
    plt.ylabel("코호트(첫 구매월)")
    plt.tick_params(axis="x", rotation=0)
    plt.show()
else:
    print("리텐션 데이터 부족")

## 4. 코호트별 LTV(ARPU) 요약

In [ ]:
ltv = first_order.groupby("cohort_month").agg(
    고객수=("customer_unique_id", "count"),
    평균_주문수=("order_count", "mean"),
    평균_LTV=("total_value", "mean"),
).round(2)
print(ltv.head(10))